# Sensibilidad directa e identificabilidad

**Nivel 3 — Crítica de investigación**  
**Prerrequisitos:** niveles 1–2 y fundamentos de inversión; 35 min.

## Objetivos

Siga la evidencia desde el contrato fuente hasta el resultado. Declare unidades y supuestos, reproduzca el cálculo e identifique qué inferencia no respalda la evidencia.

## Preparación

Usa módulos y registros versionados desde la raíz descubierta; no descarga ni modifica fuentes. Instale versiones exactas: `pip install numpy==2.2.6 matplotlib==3.11.2` `scipy==1.15.2` es requerido por el operador directo MT.

## Datos

Registro medido solo QC y artefacto sintético de calibración separado; no mezcle estas clases de evidencia.

## Método

El operador estratificado del repositorio usa respuesta de semiespacio `Z=√(iωμ₀ρ)` y la propaga recursivamente por capas finitas usando `k=√(iωμ₀/ρ)`, `w=√(iωμ₀ρ)` y `t=tanh(kh)`. La base es infinita; la geometría es 1D, isótropa y cuasiestática. Use las funciones existentes `electromagnetics.impedance` y `curves`, sin reimplementarlas. Compare la verdad sintética exacta con un ajuste guardado sobre las frecuencias versionadas, graficando resistividad aparente (Ω·m) y fase (grados) frente al período (s). Curvas parecidas no prueban unicidad de resistividad o espesor: dimensionalidad, static shift, errores correlacionados y espesor fijo quedan fuera de la calibración.

Los parámetros son log-resistividades: positivos por construcción y pasos iguales son cambios multiplicativos iguales. Residuos complejos usan sigma por componente real; ambas partes aportan al ajuste. Frecuencias no son muestras independientes de profundidad: kernels se solapan y un período no identifica una interfaz. Sensibilidad local y cobertura dependen de parametrización fija y no descartan equivalencia global.

## Estado del arte

El estado del arte no es una tabla de posiciones: separe formulación publicada, implementación del producto y evidencia versionada. Una afirmación de avance requiere dominios retenidos independientes, incertidumbre calibrada y réplica.

## Resultados

Lea ejes físicos y unidades; distinga ajuste, error contra verdad sintética y acuerdo con etiquetas humanas. El recálculo debe concordar con el recibo. Un modelo graficado no demuestra unicidad, transferencia a campo ni resolución.

### Diagnostic 1

Inspeccione el recibo y procedencia del caso.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib', 'scipy']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Call the repository layered-earth forward operator on its synthetic calibration target.
import json,sys
from pathlib import Path
sys.path.insert(0,str(root/"data-pipeline")); from electromagnetics import impedance,curves
cal=json.loads((root/"data/derived/v2/edi/calibration-two-layer.json").read_text(encoding="utf-8")); f=cal["frequencies_hz"]; rho=cal["calibration_model"]; h=cal["fixed_thickness_m"]; z=impedance(rho,h,f); truth=curves(z,f)
print("synthetic rho Ω·m",rho,"fixed thickness m",h,"n frequencies",len(f))

### Diagnostic 2

Calcule la comparación y grafíquela con unidades físicas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Forward-project a stored fitted model and compare physical response curves.
import matplotlib.pyplot as plt
fit=cal["rows"][0]["model"]; other=curves(impedance(fit,h,f),f); T=1/np.asarray(f)
fig,ax=plt.subplots(2,1,figsize=(8,6),sharex=True,constrained_layout=True); ax[0].loglog(T,truth["apparent"],"o-",label="truth"); ax[0].loglog(T,other["apparent"],"s--",label="stored fit"); ax[1].semilogx(T,truth["phase"],"o-",label="truth"); ax[1].semilogx(T,other["phase"],"s--",label="stored fit")
ax[0].set_ylabel("Apparent resistivity (Ω·m)"); ax[1].set_ylabel("Phase (degrees)"); ax[1].set_xlabel("Period (s)"); [a.grid(True,which="both",alpha=.2) or a.legend() for a in ax]; ax[0].set_title("Synthetic calibration only"); plt.show(); print("stored fitted rho Ω·m",fit)

## Resumen

La conclusión depende de geometría, adquisición, ruido, partición, parametrización y derechos de fuente. Conserve negativos y nombre la observación o control siguiente.

## Referencias

- [https://github.com/MTgeophysics/mtpy-v2](https://github.com/MTgeophysics/mtpy-v2)
- [https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california](https://www.usgs.gov/data/magnetotelluric-data-clear-lake-region-northern-california)